# Custom PTQ: ResNet-18 CIFAR-10 Head

This notebook retrains only the final classifier layer of an ImageNet-pretrained ResNet-18 for CIFAR-10, producing 10 output predictions. It then runs the custom PTQ path on that CIFAR-10 head model.


## Setup

In [91]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/sahil/Group Studies/Quantization-Group-Studies')

In [92]:
import importlib

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

model_module = importlib.import_module(
    "src.models.cnn_based.pretrained_resnet18"
)
model_module = importlib.reload(model_module)

CIFAR10_CLASS_NAMES = model_module.CIFAR10_CLASS_NAMES
build_resnet18_cifar10_classifier = model_module.build_resnet18_cifar10_classifier

custom_ptq_module = importlib.import_module(
    "src.quantization.custom_quantization.custom_ptq"
)
custom_ptq_module = importlib.reload(custom_ptq_module)

custom_ptq = custom_ptq_module.custom_ptq
dequantize_tensor = custom_ptq_module.dequantize_tensor


## Load CIFAR-10 Dataset

CIFAR-10 images and labels are used to train the new 10-class classifier head. The ResNet-18 backbone starts from ImageNet-pretrained weights and stays frozen by default.


In [93]:
(cifar_train_images, cifar_train_labels), (cifar_test_images, cifar_test_labels) = keras.datasets.cifar10.load_data()

num_head_training_samples = len(cifar_train_images)
num_test_samples = len(cifar_test_images)
num_calibration_samples = 100
batch_size = 64
head_training_epochs = 20
validation_samples = 5000

train_raw_images = cifar_train_images[:num_head_training_samples]
train_labels = cifar_train_labels[:num_head_training_samples].reshape(-1)
test_raw_images = cifar_test_images[:num_test_samples]
test_labels = cifar_test_labels[:num_test_samples].reshape(-1)

train_raw_images.shape, train_labels.shape, test_raw_images.shape, test_labels[:10]


((500, 224, 224, 3),
 (500,),
 100,
 (1, 224, 224, 3),
 array([3, 8, 8, 0, 6, 6, 1, 6, 3, 1], dtype=uint8))

## Build and Train the CIFAR-10 Head

The original ImageNet model predicts 1000 classes. Here we reuse its frozen backbone and replace the final Dense layer with a new 10-class CIFAR-10 Dense layer. To make full-dataset CPU training practical, frozen features are extracted once and the new head is trained on those features.


In [94]:
model = build_resnet18_cifar10_classifier(freeze_backbone=True)
model.summary()
backbone = model.get_layer("res_net_backbone")
pooler = model.get_layer("pooler")
head = model.get_layer("cifar10_predictions")

def extract_features(images, batch_size=batch_size):
    feature_batches = []
    dataset = tf.data.Dataset.from_tensor_slices(images).batch(batch_size)
    for image_batch in dataset:
        processed = model.preprocessor(image_batch)
        feature_map = backbone(processed, training=False)
        feature_batches.append(pooler(feature_map).numpy())
    return np.concatenate(feature_batches, axis=0)

train_features = extract_features(train_raw_images)
test_features = extract_features(test_raw_images)

head_model = keras.Sequential([
    keras.Input(shape=(train_features.shape[-1],)),
    keras.layers.Dense(len(CIFAR10_CLASS_NAMES), name="cifar10_predictions"),
])
head_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

history = head_model.fit(
    train_features,
    train_labels,
    epochs=head_training_epochs,
    batch_size=256,
    validation_split=validation_samples / num_head_training_samples,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )],
    verbose=1,
)
head.set_weights(head_model.layers[-1].get_weights())

test_loss, test_accuracy = head_model.evaluate(
    test_features, test_labels, batch_size=256, verbose=0
)
accuracy_table = pd.DataFrame([{
    "model": "fp32_resnet18_cifar10",
    "train_samples": num_head_training_samples,
    "test_samples": num_test_samples,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_accuracy_percent": float(test_accuracy * 100),
}])
accuracy_table

samples = [
    np.asarray(model.preprocessor(image[None, ...]), dtype=np.float32)
    for image in test_raw_images[:num_calibration_samples]
]


Model: "resnet18_cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ images (InputLayer)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_net_backbone                │ (None, 7, 7, 512)      │    11,186,112 │
│ (ResNetBackbone)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pooler (GlobalAveragePooling2D) │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_dropout (Dropout)        │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cifar10_predictions (Dense)     │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,191,242 (42.69 MB)

 Trainable params: 5,130 (20.04 KB)

 Non-trainable params: 11,186,112 (42.67 MB)

Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 171ms/step - accuracy: 0.2200 - loss: 2.1913 - val_accuracy: 0.3000 - val_loss: 2.0951
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 165ms/step - accuracy: 0.4775 - loss: 1.8881 - val_accuracy: 0.4600 - val_loss: 1.8885
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step - accuracy: 0.5675 - loss: 1.6637 - val_accuracy: 0.5200 - val_loss: 1.7176


((None, 10), (1, 224, 224, 3), dtype('float32'))

## Run Custom PTQ

The custom implementation uses symmetric signed INT8 quantization for weight tensors with rank 2 or higher:

```text
scale = max(abs(weight)) / 127
q = round(weight / scale)
```

For activations, the notebook runs representative CIFAR-10 samples through the 10-class model and records min/max ranges for visible intermediate outputs.


In [96]:
quantize_min_rank = 2
per_channel = True
results = custom_ptq(
    model,
    samples,
    quantize_min_rank=quantize_min_rank,
    per_channel=per_channel,
)

weight_result = results["weights"]
activation_result = results["activations"]

len(weight_result.tensors), len(activation_result.ranges)


ValueError: Output with path `1` is not connected to `inputs`

In [ ]:
# Check whether custom 8-bit weight PTQ preserves the FP32 CIFAR-10 model prediction.
# The custom quantizer leaves rank-1 tensors unchanged and dequantizes rank-2+ tensors into a copied model.
quantized_tensor_iter = iter(weight_result.tensors)
dequantized_weights = []

for weight in model.weights:
    weight_array = weight.numpy()
    should_dequantize = (
        np.issubdtype(weight_array.dtype, np.floating)
        and weight_array.ndim >= quantize_min_rank
    )

    if should_dequantize:
        quantized_tensor = next(quantized_tensor_iter)
        dequantized_weights.append(
            dequantize_tensor(quantized_tensor).astype(weight_array.dtype)
        )
    else:
        dequantized_weights.append(weight_array)

custom_dequantized_model = build_resnet18_cifar10_classifier(freeze_backbone=True)
custom_dequantized_model.set_weights(dequantized_weights)

fp32_output = model(samples[0], training=False).numpy()
custom_output = custom_dequantized_model(samples[0], training=False).numpy()
fp32_top_class = int(np.argmax(fp32_output[0]))
custom_top_class = int(np.argmax(custom_output[0]))

pd.DataFrame([
    {
        "model": "custom_8bit_dequantized",
        "dataset": "cifar10",
        "true_cifar10_label": int(test_labels[0]),
        "true_cifar10_name": CIFAR10_CLASS_NAMES[int(test_labels[0])],
        "fp32_top_class_index": fp32_top_class,
        "fp32_top_class_name": CIFAR10_CLASS_NAMES[fp32_top_class],
        "custom_top_class_index": custom_top_class,
        "custom_top_class_name": CIFAR10_CLASS_NAMES[custom_top_class],
        "matches_fp32_prediction": custom_top_class == fp32_top_class,
        "max_abs_output_difference": float(np.max(np.abs(fp32_output - custom_output))),
    }
])


## Summary Metrics

In [ ]:
summary_table = pd.DataFrame([
    {
        "target": "weights",
        "dataset": "cifar10",
        "num_head_training_samples": num_head_training_samples,
        "num_calibration_samples": len(samples),
        "quantized_tensor_min_rank": quantize_min_rank,
        "per_channel": per_channel,
        "fp32_size_bytes": weight_result.fp32_size_bytes,
        "int8_size_bytes": weight_result.int8_size_bytes,
        "compression_ratio": weight_result.compression_ratio,
        "memory_reduction_%": weight_result.memory_reduction_percent,
    },
    {
        "target": "activations",
        "dataset": "cifar10",
        "num_head_training_samples": num_head_training_samples,
        "num_calibration_samples": len(samples),
        "quantized_tensor_min_rank": None,
        "per_channel": None,
        "fp32_size_bytes": activation_result.fp32_size_bytes,
        "int8_size_bytes": activation_result.int8_size_bytes,
        "compression_ratio": activation_result.compression_ratio,
        "memory_reduction_%": activation_result.memory_reduction_percent,
    },
])

summary_table


## Weight Tensor Report

In [ ]:
weight_table = pd.DataFrame([
    {
        "name": tensor.name,
        "shape": tensor.original_shape,
        "dtype": tensor.original_dtype,
        "quantization_axis": tensor.quantization_axis,
        "scale_count": int(np.size(tensor.scale)),
        "scale_min": float(np.min(tensor.scale)),
        "scale_max": float(np.max(tensor.scale)),
        "int8_bytes": tensor.values.nbytes,
    }
    for tensor in weight_result.tensors
])

weight_table.head(10)


## Activation Calibration Report

In [ ]:
activation_table = pd.DataFrame([
    {
        "layer": layer_name,
        "min": activation_range[0],
        "max": activation_range[1],
        "scale": activation_result.scales[layer_name],
    }
    for layer_name, activation_range in activation_result.ranges.items()
])

activation_table

## Save Summary Results

In [ ]:
output_dir = PROJECT_ROOT / "artifacts" / "resnet18_custom_ptq_notebook"
output_dir.mkdir(parents=True, exist_ok=True)
model_weights_path = output_dir / "resnet18_cifar10.weights.h5"
model.save_weights(model_weights_path)

summary = {
    "dataset": "cifar10",
    "num_head_training_samples": num_head_training_samples,
    "num_test_samples": num_test_samples,
    "num_calibration_samples": len(samples),
    "model": "resnet18_cifar10_classifier",
    "num_output_classes": len(CIFAR10_CLASS_NAMES),
    "class_names": list(CIFAR10_CLASS_NAMES),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_accuracy_percent": float(test_accuracy * 100),
    "weights": {
        "num_tensors": len(weight_result.tensors),
        "quantized_tensor_min_rank": quantize_min_rank,
        "per_channel": per_channel,
        "fp32_size_bytes": weight_result.fp32_size_bytes,
        "int8_size_bytes": weight_result.int8_size_bytes,
        "compression_ratio": weight_result.compression_ratio,
        "memory_reduction_percent": weight_result.memory_reduction_percent,
    },
    "activations": {
        "num_layers": len(activation_result.ranges),
        "fp32_size_bytes": activation_result.fp32_size_bytes,
        "int8_size_bytes": activation_result.int8_size_bytes,
        "compression_ratio": activation_result.compression_ratio,
        "memory_reduction_percent": activation_result.memory_reduction_percent,
        "ranges": activation_result.ranges,
        "scales": activation_result.scales,
    },
}

results_path = output_dir / "results.json"
results_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
results_path, model_weights_path


## Notes

- The original ImageNet ResNet-18 head has 1000 outputs; this notebook replaces it with a 10-output CIFAR-10 head.
- Only the final classifier head is trained by default; the pretrained backbone stays frozen.
- Weight results quantize rank-2-or-higher tensors, such as convolution/dense kernels, with per-output-channel INT8 scales while keeping rank-1 tensors unchanged for prediction stability.
- The prediction check dequantizes custom INT8 weights into a copied CIFAR-10 model and compares its top class with the trained FP32 CIFAR-10 model on the same sample.
